In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import random
import pandas as pd
import numpy as np
import pydicom
import cv2
from tqdm import tqdm


In [ ]:
# Load training CSV
df = pd.read_csv('/kaggle/input/vinbigdata-chest-xray-abnormalities-detection/train.csv')

# Get unique image IDs
unique_images = df['image_id'].unique()

# Randomly select 1500 images
selected_images = random.sample(list(unique_images), 1500)

print("Selected images:", len(selected_images))


In [ ]:
os.makedirs("train_jpg", exist_ok=True)


In [ ]:
input_folder = "/kaggle/input/vinbigdata-chest-xray-abnormalities-detection/train"

for image_id in tqdm(selected_images):
    
    dicom_path = os.path.join(input_folder, image_id + ".dicom")
    
    # Read DICOM file
    dicom = pydicom.dcmread(dicom_path)
    image = dicom.pixel_array
    
    # Convert to 8-bit format (required for JPG)
    image = image.astype(np.float32)
    image = (image - image.min()) / (image.max() - image.min())
    image = (image * 255).astype(np.uint8)
    
    # Save as JPG
    output_path = os.path.join("train_jpg", image_id + ".jpg")
    cv2.imwrite(output_path, image)

print("Conversion completed successfully")

In [ ]:
import os

jpg_folder = "/kaggle/working/train_jpg"

print("Total JPG files:", len(os.listdir(jpg_folder)))


In [ ]:
print("Selected images count:", len(selected_images))


In [ ]:
import os

converted_files = os.listdir("/kaggle/working/train_jpg")
converted_ids = [f.replace(".jpg", "") for f in converted_files]

missing = set(selected_images) - set(converted_ids)

print("Missing images count:", len(missing))


In [ ]:
import os

source_folder = "/kaggle/input/vinbigdata-chest-xray-abnormalities-detection/train"

existing = []
not_existing = []

for image_id in selected_images:
    path = os.path.join(source_folder, image_id + ".dicom")
    if os.path.exists(path):
        existing.append(image_id)
    else:
        not_existing.append(image_id)

print("Existing DICOM files:", len(existing))
print("Missing DICOM files:", len(not_existing))


In [ ]:
import os

converted_files = os.listdir("/kaggle/working/train_jpg")
converted_ids = [f.replace(".jpg", "") for f in converted_files]

missing_images = list(set(selected_images) - set(converted_ids))

print("Images left to convert:", len(missing_images))


In [ ]:
import pydicom
import cv2
import numpy as np
from tqdm import tqdm
import os

input_folder = "/kaggle/input/vinbigdata-chest-xray-abnormalities-detection/train"

for image_id in tqdm(missing_images):
    
    dicom_path = os.path.join(input_folder, image_id + ".dicom")
    
    dicom = pydicom.dcmread(dicom_path)
    image = dicom.pixel_array
    
    image = image.astype(np.float32)
    image = (image - image.min()) / (image.max() - image.min())
    image = (image * 255).astype(np.uint8)
    
    output_path = os.path.join("/kaggle/working/train_jpg", image_id + ".jpg")
    cv2.imwrite(output_path, image)

print("Remaining images converted successfully")


In [ ]:
print("Final JPG count:", len(os.listdir("/kaggle/working/train_jpg")))


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

sample_image = os.listdir(jpg_folder)[0]
img = Image.open(os.path.join(jpg_folder, sample_image))

plt.imshow(img, cmap='gray')
plt.title(sample_image)
plt.axis('off')
plt.show()
